# Config Awal

In [1]:
import os
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
from openpyxl import Workbook
from datetime import datetime

In [4]:
# ==============================================================================
# DIRECTORY CONFIGURATION & INITIALIZATION
# Menentukan path utama, struktur folder dataset, dan file output
# ==============================================================================

# Root directory dan file input
BASE_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code"
DATASET_FOLDER = "dataset(2)"
OUTPUT_FOLDER = "output(2)"

INPUT_GITHUB = os.path.join(BASE_DIR, "asli", "nim_github.txt")

# Struktur folder pipeline
DIRS = {
    "DATASET"      : os.path.join(BASE_DIR, DATASET_FOLDER),

    # Preprocessing
    "RAW"          : os.path.join(BASE_DIR, DATASET_FOLDER, "01_Raw"),
    "NORM"         : os.path.join(BASE_DIR, DATASET_FOLDER, "02_Normalized"),
    "CONV"         : os.path.join(BASE_DIR, DATASET_FOLDER, "03_Converted"),
    "CLEAN"        : os.path.join(BASE_DIR, DATASET_FOLDER, "04_Cleaned"),

    # Formatter experiment
    "AUTOPEP8"     : os.path.join(BASE_DIR, DATASET_FOLDER, "05a_Autopep8"),
    "BLACK"        : os.path.join(BASE_DIR, DATASET_FOLDER, "05b_Black"),
    
    "FILTERED"     : os.path.join(BASE_DIR, DATASET_FOLDER, "05c_Filtered"),

    # AST & Graph
    "AST"          : os.path.join(BASE_DIR, DATASET_FOLDER, "06_AST"),
    "AST_VISUAL"   : os.path.join(BASE_DIR, DATASET_FOLDER, "06a_AST_visual"),
    "GRAPH"        : os.path.join(BASE_DIR, DATASET_FOLDER, "07_Graph"),
    "INPUT_GRAPH"  : os.path.join(BASE_DIR, DATASET_FOLDER, "08_Graph2vec_Input"),
    "EMBEDDING"    : os.path.join(BASE_DIR, DATASET_FOLDER, "09_Graph2vec_Embedding"),

    # Output
    "LOGS"         : os.path.join(BASE_DIR, OUTPUT_FOLDER),
    "RUNTIME_A"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_autopep8"),
    "RUNTIME_B"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_black"),
    # folder output untuk skor kemiripan (COSINE SIMILARITY)
    "SIMILARITY"   : os.path.join(BASE_DIR, OUTPUT_FOLDER,"similarity"),
}

# File output penelitian
RESULTS = {
    # Preprocessing
    "CLONE_REPORT"    : os.path.join(DIRS["LOGS"], "01_clone_report.xlsx"),
    "NORM_REPORT"     : os.path.join(DIRS["LOGS"], "02_normalization_report.xlsx"),
    "CONV_REPORT"     : os.path.join(DIRS["LOGS"], "03_conversion_report.xlsx"),
    "CLEAN_REPORT"    : os.path.join(DIRS["LOGS"], "04_cleaning_report.xlsx"),

    # Formatter
    "ERR_AUTOPEP"     : os.path.join(DIRS["LOGS"], "05a_autopep_errors.json"),
    "ERR_BLACK"       : os.path.join(DIRS["LOGS"], "05b_black_errors.json"),

    # Statistik kode
    "LOC_REPORT"      : os.path.join(DIRS["LOGS"], "06_loc_report.xlsx"),

    # Runtime Autopep8
    "RUN_PROJECT_A"   : os.path.join(DIRS["RUNTIME_A"], "07a_runtime_project_AUTOPEP8.xlsx"),
    "RUN_FUNCTION_A"  : os.path.join(DIRS["RUNTIME_A"], "07b_runtime_function_AUTOPEP8.xlsx"),
    "RUN_COMPARE_A"   : os.path.join(DIRS["RUNTIME_A"], "07c_runtime_compare_AUTOPEP8.xlsx"),

    # Runtime Black
    "RUN_PROJECT_B"   : os.path.join(DIRS["RUNTIME_B"], "07a_runtime_project_BLACK.xlsx"),
    "RUN_FUNCTION_B"  : os.path.join(DIRS["RUNTIME_B"], "07b_runtime_function_BLACK.xlsx"),
    "RUN_COMPARE_B"   : os.path.join(DIRS["RUNTIME_B"], "07c_runtime_compare_BLACK.xlsx"),

    # Submission
    "SUBMISSION"      : os.path.join(DIRS["LOGS"], "submission_report.xlsx"),

    # AST & Graph
    "EXTRACT_AST"     : os.path.join(DIRS["LOGS"], "09a_AST_report.xlsx"),
    "CONSTRUCT_GRAPH" : os.path.join(DIRS["LOGS"], "09b_Graph_report.xlsx"),
    "LIST_GRAPH"      : os.path.join(DIRS["LOGS"], "09c_List_Graph_report.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT": os.path.join(DIRS["LOGS"], "10_embedding_report.xlsx"),
    "EMBEDDING_VECTOR": os.path.join(DIRS["LOGS"], "10a_embedding_vector.xlsx"),

    # Similarity
    "SIMILARITY"      : os.path.join(DIRS["LOGS"], "11_similarity_report.xlsx"),
    "SIMILARITY_MODUL" : os.path.join(DIRS['LOGS'], "11a_Similarity_per_Modul.xlsx"),
    "SIMILARITY_SUMMARY" : os.path.join(DIRS['LOGS'], "11b_similarity_summary.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN" : os.path.join(DIRS['LOGS'], "12_euclidean_similarity.xlsx"),
    "EUCLIDEAN_MODUL" : os.path.join(DIRS['LOGS'], "12a_euclidean_similarity_modul.xlsx"),
    
    "SIMILARITY_COMPARE" : os.path.join(DIRS['LOGS'], "13_similarity_comparison.xlsx"),
    
    # METRICS
    "METRICS"         : os.path.join(DIRS["LOGS"], "metrics_evaluation_report.xlsx"),
}

# VALIDATION & DIRECTORY INITIALIZATION
print("=" * 70)
print(f"{'DIRECTORY CONFIGURATION':^70}")
print("=" * 70)

# Validasi root directory
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"❌ BASE_DIR tidak ditemukan: {BASE_DIR}")

# Validasi file input
if not os.path.isfile(INPUT_GITHUB):
    raise FileNotFoundError(f"❌ File input tidak ditemukan: {INPUT_GITHUB}")

if not INPUT_GITHUB.endswith(".txt"):
    raise ValueError("❌ File input harus berekstensi .txt")

if os.path.getsize(INPUT_GITHUB) == 0:
    raise ValueError(f"❌ File input kosong: {INPUT_GITHUB}")

# Membuat folder pipeline
folders_created = 0

for name, path in DIRS.items():
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        folders_created += 1
        status = "[NEW]"
    elif not os.path.isdir(path):
        raise NotADirectoryError(f"❌ Path bukan folder: {path}")
    else:
        status = "[EXISTS]"
    print(f"{status:<10} {name:<12} : {path}")

# Validasi parent folder output
missing_results_parent = []

for name, path in RESULTS.items():
    parent_dir = os.path.dirname(path)
    if not os.path.exists(parent_dir):
        missing_results_parent.append(
            f"{name} : {parent_dir}"
        )

if missing_results_parent:
    raise FileNotFoundError(
        "❌ Parent folder RESULTS tidak ditemukan:\n"
        + "\n".join(missing_results_parent)
    )

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("-" * 70)
print(f"Folders Created  : {folders_created}")
print(f"Dataset Root     : {DIRS['DATASET']}")
print(f"Input File       : {INPUT_GITHUB}")
print("=" * 70)
print(f"Diproses pada {timestamp}")

                       DIRECTORY CONFIGURATION                        
[EXISTS]   DATASET      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)
[EXISTS]   RAW          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\01_Raw
[EXISTS]   NORM         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\02_Normalized
[EXISTS]   CONV         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\03_Converted
[EXISTS]   CLEAN        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
[EXISTS]   AUTOPEP8     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05a_Autopep8
[EXISTS]   BLACK        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05b_Black
[EXISTS]   FILTERED     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05c_Filtered
[EXISTS]   AST          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_AST
[EXISTS]   AST_VISUAL   : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06a_AST_visual
[EXISTS]   GRAPH        : D:\PUTRI\D4\SEMESTER

# Perbandingan Hasil Cosine Similarity vs Euclidean Distance

In [5]:
# ── Load source data ──────────────────────────────────────────────────────────
cosine_all   = pd.read_excel(RESULTS['SIMILARITY'],   sheet_name="all_similarity")
cosine_mod   = pd.read_excel(RESULTS['SIMILARITY'],   sheet_name="modul_summary")
eucl_all     = pd.read_excel(RESULTS['EUCLIDEAN'], sheet_name="all_similarity")
eucl_mod     = pd.read_excel(RESULTS['EUCLIDEAN'], sheet_name="modul_summary")

# ── Helper styles ─────────────────────────────────────────────────────────────
HEADER_FILL_COSINE  = PatternFill("solid", fgColor="1F4E79")
HEADER_FILL_EUCLID  = PatternFill("solid", fgColor="375623")
HEADER_FILL_COMPARE = PatternFill("solid", fgColor="7B2D8B")
HEADER_FILL_DIST    = PatternFill("solid", fgColor="833C00")
HEADER_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=10)

SUBHEADER_FILLS = {
    "cosine":  PatternFill("solid", fgColor="2E75B6"),
    "euclid":  PatternFill("solid", fgColor="548235"),
    "compare": PatternFill("solid", fgColor="9B59B6"),
    "dist":    PatternFill("solid", fgColor="C55A11"),
}
SUBHEADER_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=10)

ROW_ALT  = PatternFill("solid", fgColor="EEF4FB")
ROW_ALT2 = PatternFill("solid", fgColor="EFF7EE")
ROW_ALT_COMPARE = PatternFill("solid", fgColor="F5EFF7")

CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",   vertical="center")

thin_side = Side(style="thin", color="BBBBBB")
THIN_BORDER = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

KATEGORI_FILL = {
    "Sangat Tinggi": PatternFill("solid", fgColor="FF0000"),
    "Tinggi":        PatternFill("solid", fgColor="FFC000"),
    "Sedang":        PatternFill("solid", fgColor="FFFF00"),
    "Rendah":        PatternFill("solid", fgColor="92D050"),
}
KATEGORI_FONT = {
    "Sangat Tinggi": Font(bold=True, color="FFFFFF", name="Arial", size=9),
    "Tinggi":        Font(bold=True, color="000000", name="Arial", size=9),
    "Sedang":        Font(bold=True, color="000000", name="Arial", size=9),
    "Rendah":        Font(bold=True, color="000000", name="Arial", size=9),
}

def set_header(cell, text, fill, font=HEADER_FONT):
    cell.value = text
    cell.font = font
    cell.fill = fill
    cell.alignment = CENTER
    cell.border = THIN_BORDER

def set_subheader(cell, text, color_key):
    cell.value = text
    cell.font = SUBHEADER_FONT
    cell.fill = SUBHEADER_FILLS[color_key]
    cell.alignment = CENTER
    cell.border = THIN_BORDER

def set_cell(cell, value, alt=False, fill=None, num_fmt=None, center=False):
    cell.value = value
    cell.font = Font(name="Arial", size=9)
    if fill:
        cell.fill = fill
    elif alt:
        cell.fill = ROW_ALT
    cell.alignment = CENTER if center else LEFT
    cell.border = THIN_BORDER
    if num_fmt:
        cell.number_format = num_fmt

# ── Build workbook ────────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# ═════════════════════════════════════════════════════════════════════════════
# SHEET 1 – Perbandingan Per Modul
# ═════════════════════════════════════════════════════════════════════════════
ws1 = wb.create_sheet("Perbandingan Modul")
ws1.freeze_panes = "A3"

# Merge + main header
ws1.merge_cells("A1:I1")
set_header(ws1["A1"], "PERBANDINGAN SKOR KEMIRIPAN KODE — COSINE vs EUCLIDEAN (PER MODUL)",
           HEADER_FILL_COMPARE, Font(bold=True, color="FFFFFF", name="Arial", size=12))

# Sub-headers row 2
headers = [
    ("A2", "Modul",                 "compare"),
    ("B2", "Cosine\nScore (%)",     "cosine"),
    ("C2", "Cosine\nRank",          "cosine"),
    ("D2", "Cosine\nKategori",      "cosine"),
    ("E2", "Euclidean\nScore (%)",  "euclid"),
    ("F2", "Euclidean\nRank",       "euclid"),
    ("G2", "Euclidean\nKategori",   "euclid"),
    ("H2", "Selisih\nSkor (%)",     "compare"),
    ("I2", "Keterangan",            "compare"),
]
for cell_ref, text, color in headers:
    set_subheader(ws1[cell_ref], text, color)

# Merge modul data
cosine_mod_s  = cosine_mod.sort_values("modul").reset_index(drop=True)
eucl_mod_s    = eucl_mod.sort_values("modul").reset_index(drop=True)
merged_mod    = cosine_mod_s.merge(eucl_mod_s, on="modul", suffixes=("_cos", "_euc"))
merged_mod["rank_cos"] = merged_mod["similarity_score_cos"].rank(ascending=False).astype(int)
merged_mod["rank_euc"] = merged_mod["similarity_score_euc"].rank(ascending=False).astype(int)
merged_mod["selisih"]  = merged_mod["similarity_score_cos"] - merged_mod["similarity_score_euc"]

def kategori_cos(v):
    if v >= 80: return "Sangat Tinggi"
    if v >= 60: return "Tinggi"
    if v >= 40: return "Sedang"
    return "Rendah"

def kategori_euc(v):
    if v >= 70: return "Sangat Tinggi"
    if v >= 50: return "Tinggi"
    if v >= 30: return "Sedang"
    return "Rendah"

merged_mod["kat_cos"] = merged_mod["similarity_score_cos"].apply(kategori_cos)
merged_mod["kat_euc"] = merged_mod["similarity_score_euc"].apply(kategori_euc)

for i, row in merged_mod.iterrows():
    r = i + 3
    alt = (i % 2 == 0)
    sel = row["selisih"]
    keterangan = "Cosine lebih tinggi" if sel > 0 else ("Euclidean lebih tinggi" if sel < 0 else "Sama")

    set_cell(ws1.cell(r, 1), row["modul"],                  alt, center=True)
    set_cell(ws1.cell(r, 2), round(row["similarity_score_cos"], 2), alt, num_fmt="0.00", center=True)
    set_cell(ws1.cell(r, 3), row["rank_cos"],               alt, center=True)
    kat_c = row["kat_cos"]
    ws1.cell(r, 4).value = kat_c
    ws1.cell(r, 4).fill  = KATEGORI_FILL.get(kat_c, PatternFill())
    ws1.cell(r, 4).font  = KATEGORI_FONT.get(kat_c, Font(name="Arial", size=9))
    ws1.cell(r, 4).alignment = CENTER
    ws1.cell(r, 4).border    = THIN_BORDER

    set_cell(ws1.cell(r, 5), round(row["similarity_score_euc"], 2), alt, num_fmt="0.00", center=True)
    set_cell(ws1.cell(r, 6), row["rank_euc"],               alt, center=True)
    kat_e = row["kat_euc"]
    ws1.cell(r, 7).value = kat_e
    ws1.cell(r, 7).fill  = KATEGORI_FILL.get(kat_e, PatternFill())
    ws1.cell(r, 7).font  = KATEGORI_FONT.get(kat_e, Font(name="Arial", size=9))
    ws1.cell(r, 7).alignment = CENTER
    ws1.cell(r, 7).border    = THIN_BORDER

    sel_fill = PatternFill("solid", fgColor="DDEBF7") if sel > 0 else PatternFill("solid", fgColor="E2EFDA")
    ws1.cell(r, 8).value     = round(sel, 2)
    ws1.cell(r, 8).font      = Font(name="Arial", size=9, bold=True,
                                    color="1F4E79" if sel > 0 else "375623")
    ws1.cell(r, 8).fill      = sel_fill
    ws1.cell(r, 8).alignment = CENTER
    ws1.cell(r, 8).border    = THIN_BORDER
    ws1.cell(r, 8).number_format = "+0.00;-0.00;0.00"

    set_cell(ws1.cell(r, 9), keterangan, alt)

col_widths = [12, 15, 10, 18, 15, 12, 18, 14, 22]
for ci, w in enumerate(col_widths, 1):
    ws1.column_dimensions[get_column_letter(ci)].width = w
ws1.row_dimensions[1].height = 28
ws1.row_dimensions[2].height = 38

# ═════════════════════════════════════════════════════════════════════════════
# SHEET 2 – Distribusi Kategori
# ═════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Distribusi Kategori")
ws2.freeze_panes = "A3"

# Count per kategori
cos_dist  = cosine_all.groupby("kategori")["nim_1"].count().reset_index()
cos_dist.columns  = ["kategori", "jumlah"]
euc_dist  = eucl_all.groupby("kategori")["nim_1"].count().reset_index()
euc_dist.columns  = ["kategori", "jumlah"]

all_kat   = sorted(set(cos_dist["kategori"]) | set(euc_dist["kategori"]),
                   key=lambda x: ["Sangat Tinggi","Tinggi","Sedang","Rendah"].index(x)
                   if x in ["Sangat Tinggi","Tinggi","Sedang","Rendah"] else 99)

cos_map  = dict(zip(cos_dist["kategori"], cos_dist["jumlah"]))
euc_map  = dict(zip(euc_dist["kategori"], euc_dist["jumlah"]))
total    = len(cosine_all)

# Also per-modul kategori counts
cos_mod_kat  = cosine_all.groupby(["modul","kategori"]).size().unstack(fill_value=0)
euc_mod_kat  = eucl_all.groupby(["modul","kategori"]).size().unstack(fill_value=0)

# --- Section: overall distribution ---
ws2.merge_cells("A1:H1")
set_header(ws2["A1"], "DISTRIBUSI KATEGORI — COSINE vs EUCLIDEAN",
           HEADER_FILL_COMPARE, Font(bold=True, color="FFFFFF", name="Arial", size=12))

sub2 = [
    ("A2","Kategori","dist"),("B2","Cosine\nJumlah","cosine"),("C2","Cosine\n%","cosine"),
    ("D2","Euclidean\nJumlah","euclid"),("E2","Euclidean\n%","euclid"),
    ("F2","Selisih\nJumlah","compare"),("G2","Selisih\n%","compare"),("H2","Keterangan","compare"),
]
for ref, txt, col in sub2:
    set_subheader(ws2[ref], txt, col)

for i, kat in enumerate(all_kat):
    r   = i + 3
    alt = (i % 2 == 0)
    c_j = cos_map.get(kat, 0)
    e_j = euc_map.get(kat, 0)
    c_p = c_j / total * 100
    e_p = e_j / total * 100
    sel_j = c_j - e_j
    sel_p = c_p - e_p
    ket = "Cosine lebih banyak" if sel_j > 0 else ("Euclidean lebih banyak" if sel_j < 0 else "Sama")

    ws2.cell(r,1).value = kat
    ws2.cell(r,1).fill  = KATEGORI_FILL.get(kat, PatternFill())
    ws2.cell(r,1).font  = KATEGORI_FONT.get(kat, Font(name="Arial", size=9))
    ws2.cell(r,1).alignment = CENTER
    ws2.cell(r,1).border    = THIN_BORDER

    set_cell(ws2.cell(r,2), c_j, alt, center=True)
    set_cell(ws2.cell(r,3), round(c_p,2), alt, num_fmt="0.00\"%\"", center=True)
    set_cell(ws2.cell(r,4), e_j, alt, center=True)
    set_cell(ws2.cell(r,5), round(e_p,2), alt, num_fmt="0.00\"%\"", center=True)
    set_cell(ws2.cell(r,6), sel_j, alt, center=True)
    ws2.cell(r,6).font = Font(name="Arial", size=9, bold=True,
                              color="1F4E79" if sel_j > 0 else "375623")
    set_cell(ws2.cell(r,7), round(sel_p,2), alt, num_fmt="+0.00\"%\";-0.00\"%\";0.00\"%\"", center=True)
    ws2.cell(r,7).font = Font(name="Arial", size=9, bold=True,
                              color="1F4E79" if sel_j > 0 else "375623")
    set_cell(ws2.cell(r,8), ket, alt)

# Total row
r = len(all_kat) + 3
ws2.cell(r,1).value = "TOTAL"; ws2.cell(r,1).font = Font(bold=True, name="Arial", size=9); ws2.cell(r,1).alignment = CENTER; ws2.cell(r,1).border = THIN_BORDER
for ci, val in [(2,total),(3,100.0),(4,total),(5,100.0)]:
    ws2.cell(r,ci).value = val
    ws2.cell(r,ci).font  = Font(bold=True, name="Arial", size=9)
    ws2.cell(r,ci).alignment = CENTER
    ws2.cell(r,ci).border = THIN_BORDER
    if ci in (3,5): ws2.cell(r,ci).number_format = "0.00\"%\""

col_widths2 = [18, 14, 10, 14, 10, 14, 10, 25]
for ci, w in enumerate(col_widths2, 1):
    ws2.column_dimensions[get_column_letter(ci)].width = w
ws2.row_dimensions[1].height = 28
ws2.row_dimensions[2].height = 38

# ═════════════════════════════════════════════════════════════════════════════
# SHEET 3 – Detail Perbandingan (sample top-N per modul)
# ═════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet("Detail Perbandingan")
ws3.freeze_panes = "A3"

# Merge on nim_1, nim_2, modul, file
cos_det  = cosine_all[["modul","file","nim_1","nim_2","similarity_score","kategori"]].copy()
cos_det.columns = ["modul","file","nim_1","nim_2","cos_score","cos_kat"]
euc_det  = eucl_all[["modul","file","nim_1","nim_2","euclidean_distance","similarity_score","kategori"]].copy()
euc_det.columns = ["modul","file","nim_1","nim_2","euc_dist","euc_score","euc_kat"]

merged_det = cos_det.merge(euc_det, on=["modul","file","nim_1","nim_2"])
merged_det["selisih"] = merged_det["cos_score"] - merged_det["euc_score"]
merged_det = merged_det.sort_values(["modul","cos_score"], ascending=[True, False])

ws3.merge_cells("A1:K1")
set_header(ws3["A1"], "DETAIL PERBANDINGAN PASANGAN KODE — COSINE vs EUCLIDEAN (Semua Data)",
           HEADER_FILL_COMPARE, Font(bold=True, color="FFFFFF", name="Arial", size=12))

sub3 = [
    ("A2","No","compare"),("B2","Modul","compare"),("C2","File","compare"),
    ("D2","NIM 1","compare"),("E2","NIM 2","compare"),
    ("F2","Cosine\nScore (%)","cosine"),("G2","Cosine\nKategori","cosine"),
    ("H2","Euclidean\nScore (%)","euclid"),("I2","Euclidean\nKategori","euclid"),
    ("J2","Euclidean\nDistance","dist"),
    ("K2","Selisih Skor\n(Cos - Euc)","compare"),
]
for ref, txt, col in sub3:
    set_subheader(ws3[ref], txt, col)

for i, (_, row) in enumerate(merged_det.iterrows()):
    r   = i + 3
    alt = (i % 2 == 0)
    set_cell(ws3.cell(r,1),  i+1,                     alt, center=True)
    set_cell(ws3.cell(r,2),  row["modul"],             alt, center=True)
    set_cell(ws3.cell(r,3),  row["file"],              alt)
    set_cell(ws3.cell(r,4),  str(row["nim_1"]),        alt, center=True)
    set_cell(ws3.cell(r,5),  str(row["nim_2"]),        alt, center=True)
    set_cell(ws3.cell(r,6),  round(row["cos_score"],2),alt, num_fmt="0.00", center=True)

    kc = row["cos_kat"]
    ws3.cell(r,7).value = kc
    ws3.cell(r,7).fill  = KATEGORI_FILL.get(kc, PatternFill())
    ws3.cell(r,7).font  = KATEGORI_FONT.get(kc, Font(name="Arial",size=9))
    ws3.cell(r,7).alignment = CENTER; ws3.cell(r,7).border = THIN_BORDER

    set_cell(ws3.cell(r,8),  round(row["euc_score"],2),alt, num_fmt="0.00", center=True)

    ke = row["euc_kat"]
    ws3.cell(r,9).value = ke
    ws3.cell(r,9).fill  = KATEGORI_FILL.get(ke, PatternFill())
    ws3.cell(r,9).font  = KATEGORI_FONT.get(ke, Font(name="Arial",size=9))
    ws3.cell(r,9).alignment = CENTER; ws3.cell(r,9).border = THIN_BORDER

    set_cell(ws3.cell(r,10), round(row["euc_dist"],5), alt, num_fmt="0.00000", center=True)
    sel = row["selisih"]
    ws3.cell(r,11).value = round(sel,2)
    ws3.cell(r,11).font  = Font(name="Arial",size=9,bold=True,
                                color="1F4E79" if sel>0 else "375623" if sel<0 else "000000")
    ws3.cell(r,11).fill  = PatternFill("solid", fgColor="DDEBF7") if sel>0 else \
                            PatternFill("solid", fgColor="E2EFDA") if sel<0 else \
                            (ROW_ALT if alt else PatternFill())
    ws3.cell(r,11).alignment = CENTER; ws3.cell(r,11).border = THIN_BORDER
    ws3.cell(r,11).number_format = "+0.00;-0.00;0.00"

col_widths3 = [7, 10, 12, 14, 14, 14, 16, 14, 16, 14, 14]
for ci, w in enumerate(col_widths3, 1):
    ws3.column_dimensions[get_column_letter(ci)].width = w
ws3.row_dimensions[1].height = 28
ws3.row_dimensions[2].height = 42

# ═════════════════════════════════════════════════════════════════════════════
# SHEET 4 – Ringkasan Statistik
# ═════════════════════════════════════════════════════════════════════════════
ws4 = wb.create_sheet("Ringkasan Statistik")

ws4.merge_cells("A1:D1")
set_header(ws4["A1"], "RINGKASAN STATISTIK PERBANDINGAN",
           HEADER_FILL_COMPARE, Font(bold=True, color="FFFFFF", name="Arial", size=12))

stats = [
    ("STATISTIK GLOBAL", None, None),
    ("Metrik", "Cosine Similarity", "Euclidean Similarity"),
    ("Total Pasangan",    len(cosine_all),           len(eucl_all)),
    ("Skor Rata-rata (%)", round(cosine_all["similarity_score"].mean(),4), round(eucl_all["similarity_score"].mean(),4)),
    ("Skor Tertinggi (%)", round(cosine_all["similarity_score"].max(),4),  round(eucl_all["similarity_score"].max(),4)),
    ("Skor Terendah (%)",  round(cosine_all["similarity_score"].min(),4),  round(eucl_all["similarity_score"].min(),4)),
    ("Standar Deviasi",    round(cosine_all["similarity_score"].std(),4),  round(eucl_all["similarity_score"].std(),4)),
    ("Median (%)",         round(cosine_all["similarity_score"].median(),4),round(eucl_all["similarity_score"].median(),4)),
    (None, None, None),
    ("JUMLAH KATEGORI", None, None),
    ("Kategori", "Cosine", "Euclidean"),
]
for kat in all_kat:
    stats.append((kat, cos_map.get(kat,0), euc_map.get(kat,0)))

for i, (a, b, c) in enumerate(stats):
    r = i + 2
    if a is None and b is None:
        continue
    if b is None:  # section header
        ws4.merge_cells(f"A{r}:D{r}")
        ws4[f"A{r}"].value = a
        ws4[f"A{r}"].font  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        ws4[f"A{r}"].fill  = HEADER_FILL_COMPARE
        ws4[f"A{r}"].alignment = CENTER
        ws4[f"A{r}"].border = THIN_BORDER
    elif a in ("Metrik","Kategori"):
        for ci, val in enumerate([a, b, c], 1):
            ws4.cell(r,ci).value = val
            ws4.cell(r,ci).font  = SUBHEADER_FONT
            ws4.cell(r,ci).fill  = SUBHEADER_FILLS["compare"] if ci==1 else \
                                    SUBHEADER_FILLS["cosine"]  if ci==2 else \
                                    SUBHEADER_FILLS["euclid"]
            ws4.cell(r,ci).alignment = CENTER
            ws4.cell(r,ci).border    = THIN_BORDER
    else:
        alt = (i % 2 == 0)
        for ci, val in enumerate([a, b, c], 1):
            cell = ws4.cell(r,ci)
            cell.value = val
            cell.font  = Font(name="Arial", size=9)
            cell.fill  = ROW_ALT if alt else PatternFill()
            cell.alignment = CENTER if ci > 1 else LEFT
            cell.border = THIN_BORDER
            if isinstance(val, float):
                cell.number_format = "0.0000"

ws4.column_dimensions["A"].width = 25
ws4.column_dimensions["B"].width = 20
ws4.column_dimensions["C"].width = 20
ws4.row_dimensions[1].height = 28

# ── Save ──────────────────────────────────────────────────────────────────────
out = RESULTS["SIMILARITY_COMPARE"]
wb.save(out)
print(f"Saved: {out}")
print(f"Sheets: {wb.sheetnames}")
print(f"Total data rows (detail): {len(merged_det)}")

Saved: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\13_similarity_comparison.xlsx
Sheets: ['Perbandingan Modul', 'Distribusi Kategori', 'Detail Perbandingan', 'Ringkasan Statistik']
Total data rows (detail): 53054


# Analisis Hasil Cosine Similarity

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              GradientFill)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0. LOAD & PARSE DATA
# ─────────────────────────────────────────────
FILE1 = RESULTS['SIMILARITY']
FILE2 = RESULTS['SIMILARITY_MODUL']

r1 = pd.read_excel(FILE1, sheet_name=None)
r2 = pd.read_excel(FILE2, sheet_name=None)

df_all = r1['all_similarity'].copy()
df_all['similarity_score'] = pd.to_numeric(df_all['similarity_score'], errors='coerce')

modul_meta = {}
for m, df in r2.items():
    meta = {}
    for i in range(3):
        val = str(df.iloc[i, 0])
        if 'Jumlah Siswa' in val:
            meta['n_siswa'] = int(val.split(':')[-1].strip())
        elif 'Total Pasangan' in val:
            meta['total_pairs'] = int(val.split(':')[-1].strip())
        elif 'Rata-rata' in val:
            meta['avg_similarity'] = float(val.split(':')[-1].strip().replace('%', ''))
    data_df = df.iloc[5:].copy()
    data_df.columns = ['modul', 'file', 'nim_1', 'nim_2', 'similarity_score', 'kategori']
    data_df['similarity_score'] = pd.to_numeric(data_df['similarity_score'], errors='coerce')
    data_df = data_df.dropna(subset=['similarity_score'])
    meta['data'] = data_df
    modul_meta[m] = meta

KATEGORI_ORDER = ['Sangat Tinggi', 'Tinggi', 'Sedang', 'Rendah']
KATEGORI_COLORS = {
    'Sangat Tinggi': '#d62728',
    'Tinggi':        '#ff7f0e',
    'Sedang':        '#f0d060',
    'Rendah':        '#2ca02c',
}

# ─────────────────────────────────────────────
# 1. BUILD SUMMARY DATAFRAME
# ─────────────────────────────────────────────
rows = []
for m, info in modul_meta.items():
    data = info['data']
    total = len(data)
    cat_counts = data['kategori'].value_counts()
    row = {
        'Modul': m.upper(),
        'Jumlah Siswa': info.get('n_siswa', ''),
        'Total Pasangan': info.get('total_pairs', total),
        'Rata-rata Similarity (%)': round(info.get('avg_similarity', data['similarity_score'].mean()), 2),
        'Min (%)': round(data['similarity_score'].min(), 2),
        'Max (%)': round(data['similarity_score'].max(), 2),
        'Std Dev': round(data['similarity_score'].std(), 2),
        'Median (%)': round(data['similarity_score'].median(), 2),
        'Sangat Tinggi (≥90%)': cat_counts.get('Sangat Tinggi', 0),
        'Tinggi (75-90%)':      cat_counts.get('Tinggi', 0),
        'Sedang (50-75%)':      cat_counts.get('Sedang', 0),
        'Rendah (<50%)':        cat_counts.get('Rendah', 0),
        '% Sangat Tinggi': round(cat_counts.get('Sangat Tinggi', 0) / total * 100, 2),
    }
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary = df_summary.sort_values('Rata-rata Similarity (%)', ascending=False).reset_index(drop=True)

# Overall stats
total_all = len(df_all)
cat_all = df_all['kategori'].value_counts()

# ─────────────────────────────────────────────
# 2. WRITE SUMMARY EXCEL
# ─────────────────────────────────────────────
OUT_EXCEL = RESULTS["SIMILARITY_SUMMARY"]
wb = Workbook()

# ── STYLES ──────────────────────────────────
HDR_FILL   = PatternFill('solid', start_color='1F3864')
HDR_FONT   = Font(name='Arial', bold=True, color='FFFFFF', size=11)
SUBHDR_FILL= PatternFill('solid', start_color='2E75B6')
SUBHDR_FONT= Font(name='Arial', bold=True, color='FFFFFF', size=10)
TITLE_FONT = Font(name='Arial', bold=True, size=14, color='1F3864')
LABEL_FONT = Font(name='Arial', bold=True, size=10)
CELL_FONT  = Font(name='Arial', size=10)
BORDER_THIN= Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin'))
BORDER_MED = Border(
    left=Side(style='medium'), right=Side(style='medium'),
    top=Side(style='medium'),  bottom=Side(style='medium'))
CENTER     = Alignment(horizontal='center', vertical='center')
LEFT       = Alignment(horizontal='left',   vertical='center')

def apply_header(ws, row, col, value, fill=HDR_FILL, font=HDR_FONT):
    c = ws.cell(row=row, column=col, value=value)
    c.fill, c.font, c.alignment, c.border = fill, font, CENTER, BORDER_THIN

def style_data_cell(c, font=CELL_FONT, align=CENTER):
    c.font, c.alignment, c.border = font, align, BORDER_THIN

# ── SHEET 1: Overview ───────────────────────
ws1 = wb.active
ws1.title = 'Overview'

ws1.merge_cells('A1:M1')
ws1['A1'] = '📊 LAPORAN ANALISIS SIMILARITY — COSINE SIMILARITY'
ws1['A1'].font = TITLE_FONT
ws1['A1'].alignment = CENTER
ws1['A1'].fill = PatternFill('solid', start_color='D6E4F7')
ws1.row_dimensions[1].height = 30

ws1.merge_cells('A2:M2')
ws1['A2'] = f'Generated: {pd.Timestamp.now().strftime("%d %B %Y %H:%M")}  |  Total Pasangan: {total_all:,}  |  Total File: {len(modul_meta)}'
ws1['A2'].font = Font(name='Arial', italic=True, size=9, color='666666')
ws1['A2'].alignment = CENTER

# Overall kategori boxes (row 4-6)
ws1.row_dimensions[3].height = 8
ws1.merge_cells('A4:B4'); ws1['A4'] = 'DISTRIBUSI KATEGORI KESELURUHAN'
ws1['A4'].font = Font(name='Arial', bold=True, size=11, color='1F3864')

col_box = 1
box_info = [
    ('Sangat Tinggi\n(≥ 90%)',  cat_all.get('Sangat Tinggi',0), 'D62728', 'FDECEA'),
    ('Tinggi\n(75–90%)',        cat_all.get('Tinggi',0),        'FF7F0E', 'FFF3E0'),
    ('Sedang\n(50–75%)',        cat_all.get('Sedang',0),        'E0A800', 'FFFDE7'),
    ('Rendah\n(< 50%)',         cat_all.get('Rendah',0),        '2CA02C', 'E8F5E9'),
]
ws1.row_dimensions[5].height = 40
ws1.row_dimensions[6].height = 22

for label, count, fg, bg in box_info:
    ws1.merge_cells(start_row=5, start_column=col_box, end_row=5, end_column=col_box+2)
    c = ws1.cell(row=5, column=col_box, value=f'{label}\n{count:,}')
    c.font = Font(name='Arial', bold=True, size=11, color=fg)
    c.fill = PatternFill('solid', start_color=bg)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border = BORDER_THIN

    ws1.merge_cells(start_row=6, start_column=col_box, end_row=6, end_column=col_box+2)
    pct = ws1.cell(row=6, column=col_box, value=f'{count/total_all*100:.1f}% dari {total_all:,} pasangan')
    pct.font = Font(name='Arial', size=9, color='444444')
    pct.alignment = CENTER
    col_box += 3

# Headers row 8
ws1.row_dimensions[7].height = 8
headers = [
    'Modul', 'Jumlah Siswa', 'Total Pasangan',
    'Rata-rata (%)', 'Min (%)', 'Max (%)', 'Std Dev', 'Median (%)',
    'Sangat Tinggi', 'Tinggi', 'Sedang', 'Rendah', '% Sangat Tinggi'
]
ws1.row_dimensions[8].height = 32
for ci, h in enumerate(headers, 1):
    apply_header(ws1, 8, ci, h)

# Data rows
ALT_FILL = PatternFill('solid', start_color='EEF4FB')
for ri, row in df_summary.iterrows():
    excel_row = ri + 9
    ws1.row_dimensions[excel_row].height = 18
    alt = ri % 2 == 1
    for ci, val in enumerate(row.values, 1):
        c = ws1.cell(row=excel_row, column=ci, value=val)
        style_data_cell(c)
        if alt:
            c.fill = ALT_FILL
        # color avg similarity cell
        if ci == 4 and isinstance(val, (int, float)):
            if val >= 55:   c.fill = PatternFill('solid', start_color='FDECEA'); c.font = Font(name='Arial', size=10, color='C0392B', bold=True)
            elif val >= 45: c.fill = PatternFill('solid', start_color='FFF3E0'); c.font = Font(name='Arial', size=10, color='E67E22')
            else:           c.fill = PatternFill('solid', start_color='E8F5E9'); c.font = Font(name='Arial', size=10, color='27AE60')

# Column widths
col_widths = [10, 14, 16, 16, 10, 10, 10, 12, 16, 12, 12, 12, 18]
for i, w in enumerate(col_widths, 1):
    ws1.column_dimensions[get_column_letter(i)].width = w

# Conditional formatting for % Sangat Tinggi
last_data_row = 8 + len(df_summary)
ws1.conditional_formatting.add(
    f'M9:M{last_data_row}',
    ColorScaleRule(start_type='min', start_color='C6EFCE',
                   end_type='max',   end_color='FF0000'))

# ── SHEET 2: Per-Modul Detail ────────────────
ws2 = wb.create_sheet('Per-Modul Detail')
ws2.merge_cells('A1:H1')
ws2['A1'] = 'DETAIL STATISTIK PER MODUL'
ws2['A1'].font = TITLE_FONT
ws2['A1'].alignment = CENTER
ws2['A1'].fill = PatternFill('solid', start_color='D6E4F7')
ws2.row_dimensions[1].height = 28

r = 3
for m_info in modul_meta.items():
    m, info = m_info
    data = info['data']
    # Section header
    ws2.merge_cells(start_row=r, start_column=1, end_row=r, end_column=8)
    c = ws2.cell(row=r, column=1, value=f'MODUL: {m.upper()}')
    c.font = Font(name='Arial', bold=True, size=11, color='FFFFFF')
    c.fill = SUBHDR_FILL
    c.alignment = CENTER
    ws2.row_dimensions[r].height = 22
    r += 1

    # Stats table
    stats = [
        ('Jumlah Siswa', info.get('n_siswa', len(data['nim_1'].unique()))),
        ('Total Pasangan', info.get('total_pairs', len(data))),
        ('Rata-rata Similarity', f"{info.get('avg_similarity', data['similarity_score'].mean()):.2f}%"),
        ('Minimum', f"{data['similarity_score'].min():.2f}%"),
        ('Maksimum', f"{data['similarity_score'].max():.2f}%"),
        ('Median', f"{data['similarity_score'].median():.2f}%"),
        ('Std Deviasi', f"{data['similarity_score'].std():.2f}"),
    ]
    cat_counts = data['kategori'].value_counts()
    for kat in KATEGORI_ORDER:
        cnt = cat_counts.get(kat, 0)
        pct = cnt / len(data) * 100
        stats.append((f'  → {kat}', f'{cnt:,} ({pct:.1f}%)'))

    for label, val in stats:
        ws2.row_dimensions[r].height = 16
        lc = ws2.cell(row=r, column=1, value=label)
        lc.font = Font(name='Arial', bold=True, size=9)
        lc.alignment = LEFT
        lc.border = BORDER_THIN
        vc = ws2.cell(row=r, column=2, value=val)
        vc.font = Font(name='Arial', size=9)
        vc.alignment = CENTER
        vc.border = BORDER_THIN
        r += 1
    r += 1

ws2.column_dimensions['A'].width = 28
ws2.column_dimensions['B'].width = 20

# ── SHEET 3: Raw kategori counts ────────────
ws3 = wb.create_sheet('Kategori per Modul')
ws3.merge_cells('A1:F1')
ws3['A1'] = 'DISTRIBUSI KATEGORI PER MODUL'
ws3['A1'].font = TITLE_FONT
ws3['A1'].alignment = CENTER
ws3['A1'].fill = PatternFill('solid', start_color='D6E4F7')

hdrs3 = ['Modul', 'Total Pasangan', 'Sangat Tinggi', 'Tinggi', 'Sedang', 'Rendah']
for ci, h in enumerate(hdrs3, 1):
    apply_header(ws3, 2, ci, h)

cat_data = []
for m, info in modul_meta.items():
    data = info['data']
    cc = data['kategori'].value_counts()
    cat_data.append({
        'Modul': m.upper(),
        'Total Pasangan': len(data),
        'Sangat Tinggi': cc.get('Sangat Tinggi', 0),
        'Tinggi': cc.get('Tinggi', 0),
        'Sedang': cc.get('Sedang', 0),
        'Rendah': cc.get('Rendah', 0),
    })

df_cat = pd.DataFrame(cat_data).sort_values('Sangat Tinggi', ascending=False).reset_index(drop=True)
for ri, row in df_cat.iterrows():
    er = ri + 3
    ws3.row_dimensions[er].height = 18
    for ci, val in enumerate(row.values, 1):
        c = ws3.cell(row=er, column=ci, value=val)
        style_data_cell(c)
        if ri % 2 == 1:
            c.fill = ALT_FILL

for ci, w in enumerate([12, 16, 16, 12, 12, 12], 1):
    ws3.column_dimensions[get_column_letter(ci)].width = w

ws3.conditional_formatting.add(
    f'C3:C{2+len(df_cat)}',
    ColorScaleRule(start_type='min', start_color='C6EFCE',
                   end_type='max',   end_color='FF0000'))

wb.save(OUT_EXCEL)
print(f'✅  Excel saved → {OUT_EXCEL}')


# ─────────────────────────────────────────────
# 3. CHARTS
# ─────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8FAFB',
})

MODULS = df_summary['Modul'].tolist()
AVGS   = df_summary['Rata-rata Similarity (%)'].tolist()
CAT_COLORS_LIST = [KATEGORI_COLORS[k] for k in KATEGORI_ORDER]

# ── CHART 1: Bar chart rata-rata similarity per modul ──
fig1, ax1 = plt.subplots(figsize=(14, 6))
bar_colors = ['#C0392B' if v >= 55 else '#E67E22' if v >= 45 else '#27AE60' for v in AVGS]
bars = ax1.barh(MODULS, AVGS, color=bar_colors, height=0.65, edgecolor='white', linewidth=0.8)
ax1.axvline(x=50, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label='Threshold 50%')
for bar, val in zip(bars, AVGS):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}%', va='center', fontsize=9, fontweight='bold')
ax1.set_xlabel('Rata-rata Cosine Similarity (%)', fontsize=11)
ax1.set_title('Rata-rata Cosine Similarity per Modul', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlim(0, max(AVGS) + 8)
patches = [
    mpatches.Patch(color='#C0392B', label='≥ 55% (Tinggi)'),
    mpatches.Patch(color='#E67E22', label='45–55% (Sedang)'),
    mpatches.Patch(color='#27AE60', label='< 45% (Rendah)'),
    plt.Line2D([0],[0], color='gray', linestyle='--', label='Threshold 50%'),
]
ax1.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig((os.path.join(DIRS['SIMILARITY'], 'chart1_avg_similarity.png')), dpi=150, bbox_inches='tight')
plt.close()
print('✅  Chart 1 saved')

# ── CHART 2: Stacked bar distribusi kategori ──
fig2, ax2 = plt.subplots(figsize=(14, 6))
bottom = np.zeros(len(df_cat))
df_chart = df_cat.sort_values('Sangat Tinggi', ascending=False).reset_index(drop=True)
total_arr = df_chart['Total Pasangan'].values
for kat, color in zip(KATEGORI_ORDER, CAT_COLORS_LIST):
    vals = df_chart[kat].values / total_arr * 100
    bars = ax2.bar(df_chart['Modul'], vals, bottom=bottom, color=color,
                   label=kat, width=0.65, edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        if v > 4:
            ax2.text(bar.get_x() + bar.get_width()/2,
                     bar.get_y() + bar.get_height()/2,
                     f'{v:.1f}%', ha='center', va='center', fontsize=7.5,
                     color='white', fontweight='bold')
    bottom += vals
ax2.set_ylabel('Persentase Pasangan (%)', fontsize=11)
ax2.set_title('Distribusi Kategori Similarity per Modul (%)', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylim(0, 105)
ax2.legend(loc='upper right', fontsize=9, ncol=2)
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig((os.path.join(DIRS['SIMILARITY'], 'chart2_kategori_stacked.png')), dpi=150, bbox_inches='tight')
plt.close()
print('✅  Chart 2 saved')

# ── CHART 3: Pie chart kategori keseluruhan ──
fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(13, 6))
sizes = [cat_all.get(k, 0) for k in KATEGORI_ORDER]
colors_pie = [KATEGORI_COLORS[k] for k in KATEGORI_ORDER]
explode = (0.05, 0.03, 0.02, 0)
wedges, texts, autotexts = ax3a.pie(
    sizes, labels=KATEGORI_ORDER, colors=colors_pie, explode=explode,
    autopct=lambda p: f'{p:.1f}%\n({int(p/100*total_all):,})',
    startangle=140, pctdistance=0.78,
    textprops={'fontsize': 9},
)
for at in autotexts:
    at.set_fontsize(8.5)
    at.set_fontweight('bold')
ax3a.set_title('Distribusi Kategori Keseluruhan\n(semua modul)', fontsize=12, fontweight='bold')

# Donut per-type (js vs kelompok vs kuis)
type_groups = {
    'JS Modules': df_all[df_all['modul'].str.startswith('js')],
    'Kelompok':   df_all[df_all['modul'] == 'kelompok'],
    'Kuis':       df_all[df_all['modul'] == 'kuis'],
}
grp_avgs = [(g, df['similarity_score'].mean()) for g, df in type_groups.items()]
grp_labels, grp_vals = zip(*grp_avgs)
bar_col = ['#2E75B6', '#E67E22', '#9B59B6']
bars3 = ax3b.bar(grp_labels, grp_vals, color=bar_col, width=0.5, edgecolor='white', linewidth=1)
for bar, val in zip(bars3, grp_vals):
    ax3b.text(bar.get_x() + bar.get_width()/2, val + 0.5,
              f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')
ax3b.set_ylabel('Rata-rata Similarity (%)', fontsize=11)
ax3b.set_title('Rata-rata Similarity per Jenis\n(JS Modules / Kelompok / Kuis)', fontsize=12, fontweight='bold')
ax3b.set_ylim(0, max(grp_vals) + 10)
plt.tight_layout()
plt.savefig((os.path.join(DIRS['SIMILARITY'], 'chart3_pie_overview.png')), dpi=150, bbox_inches='tight')
plt.close()
print('✅  Chart 3 saved')

# ── CHART 4: Heatmap Sangat Tinggi per modul per file ──
file_grp = df_all.groupby(['modul', 'file'])
heat_data = file_grp.apply(
    lambda x: (x['kategori'] == 'Sangat Tinggi').sum() / len(x) * 100
).reset_index()
heat_data.columns = ['modul', 'file', 'pct_st']
heat_pivot = heat_data.pivot(index='modul', columns='file', values='pct_st').fillna(0)
heat_pivot.index = [i.upper() for i in heat_pivot.index]

fig4, ax4 = plt.subplots(figsize=(14, 7))
im = ax4.imshow(heat_pivot.values, aspect='auto', cmap='YlOrRd', vmin=0)
ax4.set_xticks(range(len(heat_pivot.columns)))
ax4.set_xticklabels(heat_pivot.columns, rotation=45, ha='right', fontsize=8)
ax4.set_yticks(range(len(heat_pivot.index)))
ax4.set_yticklabels(heat_pivot.index, fontsize=9)
for i in range(len(heat_pivot.index)):
    for j in range(len(heat_pivot.columns)):
        val = heat_pivot.values[i, j]
        if val > 0:
            ax4.text(j, i, f'{val:.0f}%', ha='center', va='center',
                     fontsize=7, color='white' if val > 40 else 'black')
plt.colorbar(im, ax=ax4, label='% Pasangan Sangat Tinggi (≥90%)')
ax4.set_title('Heatmap % Pasangan "Sangat Tinggi" per Modul × File', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig((os.path.join(DIRS['SIMILARITY'], 'chart4_heatmap.png')), dpi=150, bbox_inches='tight')
plt.close()
print('✅  Chart 4 saved')

# ── CHART 5: Box plot distribusi similarity per modul ──
modul_plot_order = df_summary['Modul'].str.lower().tolist()
plot_data = [df_all[df_all['modul'] == m]['similarity_score'].dropna().values
             for m in modul_plot_order]

fig5, ax5 = plt.subplots(figsize=(14, 6))
bp = ax5.boxplot(plot_data, patch_artist=True, vert=False,
                 medianprops={'color': 'black', 'linewidth': 2},
                 whiskerprops={'linewidth': 1.2},
                 capprops={'linewidth': 1.5},
                 flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3})
bx_colors = ['#C0392B' if a >= 55 else '#E67E22' if a >= 45 else '#27AE60' for a in AVGS]
for patch, color in zip(bp['boxes'], bx_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax5.set_yticks(range(1, len(modul_plot_order)+1))
ax5.set_yticklabels([m.upper() for m in modul_plot_order], fontsize=9)
ax5.set_xlabel('Cosine Similarity (%)', fontsize=11)
ax5.set_title('Distribusi Cosine Similarity per Modul (Box Plot)', fontsize=14, fontweight='bold', pad=15)
ax5.axvline(x=90, color='red',  linestyle='--', alpha=0.6, linewidth=1.2, label='≥90% (Sangat Tinggi)')
ax5.axvline(x=75, color='orange', linestyle='--', alpha=0.6, linewidth=1.2, label='≥75% (Tinggi)')
ax5.axvline(x=50, color='gray',   linestyle='--', alpha=0.5, linewidth=1.0, label='≥50% (Sedang)')
ax5.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig((os.path.join(DIRS['SIMILARITY'], 'chart5_boxplot.png')), dpi=150, bbox_inches='tight')
plt.close()
print('✅  Chart 5 saved')

# ── CHART 6: Top-N pasangan paling mencurigakan ──
df_top = df_all[df_all['kategori'] == 'Sangat Tinggi'].copy()
top_by_modul = df_top.groupby('modul').size().reset_index(name='count')
top_by_modul = top_by_modul.sort_values('count', ascending=True)

fig6, ax6 = plt.subplots(figsize=(10, 6))
bars6 = ax6.barh(top_by_modul['modul'].str.upper(), top_by_modul['count'],
                  color='#C0392B', height=0.6, edgecolor='white')
for bar, v in zip(bars6, top_by_modul['count']):
    ax6.text(v + 5, bar.get_y() + bar.get_height()/2,
             f'{v:,}', va='center', fontsize=9, fontweight='bold', color='#C0392B')
ax6.set_xlabel('Jumlah Pasangan', fontsize=11)
ax6.set_title('Jumlah Pasangan "Sangat Tinggi" (≥ 90%) per Modul\n(Potensi Plagiarisme)', 
              fontsize=13, fontweight='bold', pad=15)
ax6.set_xlim(0, top_by_modul['count'].max() * 1.18)
plt.tight_layout()
plt.savefig((os.path.join(DIRS['SIMILARITY'], 'chart6_sangat_tinggi_count.png')), dpi=150, bbox_inches='tight')
plt.close()
print('✅  Chart 6 saved')

print('\n🎉  Semua output selesai!')
print(f'{DIRS["SIMILARITY"]}   → similarity_summary.xlsx')
print(f'{DIRS["SIMILARITY"]}   → chart1_avg_similarity.png')
print(f'{DIRS["SIMILARITY"]}   → chart2_kategori_stacked.png')
print(f'{DIRS["SIMILARITY"]}   → chart3_pie_overview.png')
print(f'{DIRS["SIMILARITY"]}   → chart4_heatmap.png')
print(f'{DIRS["SIMILARITY"]}   → chart5_boxplot.png')
print(f'{DIRS["SIMILARITY"]}   → chart6_sangat_tinggi_count.png')

✅  Excel saved → D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\11b_similarity_summary.xlsx
✅  Chart 1 saved
✅  Chart 2 saved
✅  Chart 3 saved
✅  Chart 4 saved
✅  Chart 5 saved
✅  Chart 6 saved

🎉  Semua output selesai!
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → similarity_summary.xlsx
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → chart1_avg_similarity.png
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → chart2_kategori_stacked.png
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → chart3_pie_overview.png
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → chart4_heatmap.png
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → chart5_boxplot.png
D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\data_output\similarity   → chart6_sangat_tinggi_count.png
